# NPC EXPERIMENTAL BRAIN 

In [3]:
import numpy as np
from openai import OpenAI
from pydantic import BaseModel
from enum import Enum

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

LMSTUDIO_BASE_URL = os.environ["LMSTUDIO_BASE_URL"]
LM_API_TOKEN = os.environ["LM_API_TOKEN"]

# Identifiant EXACT du modele charge dans LM Studio (onglet Developer).
# Pour lister les ids disponibles : execute la cellule "models" plus bas.
MODEL = "google/gemma-4-e4b"

In [5]:
# client = ollama.Client()
client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key=LM_API_TOKEN)

In [6]:
models = [m.id for m in client.models.list().data]
# models

# Couche de contrat

**Suppression des biais liés à la langue** : le contrat de sortie du LLM est désormais 100 % anglais (`UP`, `DOWN`, `LEFT`, `RIGHT`), tout comme les clés de perception et le prompt (cf. section *Analyses* en fin de notebook).

In [8]:
# Constantes

VOID        = 0
PLAYER      = 1
ENEMY       = 2
GOLD        = 3
OBSTACLE    = 4
NPC         = 5

# Affichage de la carte dans le notebook (jamais fourni au LLM)
SYMBOLS = {VOID: "·", PLAYER: "👤", ENEMY: "👹", GOLD: "💰", OBSTACLE: "█"}

# Labels sémantiques fournis au LLM (perception 100 % anglais)
NAMES = {VOID: "EMPTY", PLAYER: "PLAYER", ENEMY: "ENEMY", GOLD: "GOLD", OBSTACLE: "OBSTACLE", NPC: "NPC"}

In [9]:
class Direction(str, Enum):
    UP = "UP"
    DOWN = "DOWN"
    LEFT = "LEFT"
    RIGHT = "RIGHT"

class PlayerDecision(BaseModel):
    direction: Direction
    # decisionDetails: str

MOVES = {
    "UP": (-1, 0),
    "DOWN": (1, 0),
    "LEFT": (0, -1),
    "RIGHT": (0, 1),
}

OPPOSITE = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}

# Moteur de perception

In [10]:
def localize(world_map, entity):
    positions = np.argwhere(world_map == entity)
    return positions

In [11]:
def compute_distance(entities_positions, pos_reference):
    if len(entities_positions) == 0:
        return np.array([])
    
    v = entities_positions - pos_reference
    distances = np.linalg.norm(v, axis=1)

    return np.round(distances, 2) # /!\ approx. distance

In [12]:
def adjacent_cells(world_map, player_position):
    rows, cols = world_map.shape
    r, c = player_position
    adjacent = {}
    for name, (d_row, d_column) in MOVES.items():
        new_row, new_column = r + d_row, c + d_column
        if not (0 <= new_row < rows and 0 <= new_column < cols):
            adjacent[name] = "WALL"
        else:
            adjacent[name] = NAMES.get(world_map[new_row, new_column], "?")
    return adjacent

In [13]:
def to_direction(delta_row, delta_column):
    dirs = []
    if delta_row < 0: dirs.append("UP")
    if delta_row > 0: dirs.append("DOWN")
    if delta_column < 0: dirs.append("LEFT")
    if delta_column > 0: dirs.append("RIGHT")
    return dirs

In [14]:
def perception(world_map):
    player_position = localize(world_map, PLAYER)[0]
    gold_positions = localize(world_map, GOLD)
    enemy_positions = localize(world_map, ENEMY)

    gold_dist = compute_distance(gold_positions, player_position)
    enemy_dist = compute_distance(enemy_positions, player_position)

    closest_gold_direction = (
        to_direction(*(gold_positions[np.argmin(gold_dist)] - player_position))
        if len(gold_positions) > 0 else []
    )

    return {
        "enemies_within_radius_3": int(np.sum(enemy_dist <= 3)) if len(enemy_dist) > 0 else 0,
        "enemy_distances": enemy_dist.tolist(),
        "enemy_count": len(enemy_positions),
        "closest_enemy_distance": float(np.min(enemy_dist)) if len(enemy_dist) > 0 else 999,
        "gold_distances": gold_dist.tolist(),
        "closest_gold_distance": float(np.min(gold_dist)) if len(gold_dist) > 0 else 999,
        "closest_gold_direction": closest_gold_direction,
        "adjacent_cells": adjacent_cells(world_map, player_position),
    }

In [15]:
def show_map(world_map):
    for row in world_map:
        print("\t".join(SYMBOLS.get(cell, "?") for cell in row))

# Moteur de déplacement

In [16]:
# Raisons d'échec d'un déplacement (fournies telles quelles au LLM)
BLOCK_MAP_EDGE = "MAP_EDGE"
BLOCK_ENEMY    = "ENEMY"
BLOCK_OBSTACLE = "OBSTACLE"

def blocked_reason(world_map: np.ndarray, pos):
    """Retourne None si la case est praticable, sinon la raison du blocage."""
    n_rows, n_cols = world_map.shape
    r, c = pos

    if r < 0 or c < 0 or r >= n_rows or c >= n_cols:
        return BLOCK_MAP_EDGE

    if world_map[r, c] == ENEMY:
        return BLOCK_ENEMY

    if world_map[r, c] in (VOID, GOLD):
        return None

    return BLOCK_OBSTACLE

def allowed_move(world_map: np.ndarray, pos):
    return blocked_reason(world_map, pos) is None

In [17]:
def move(world_map: np.ndarray, old_pos, new_pos):
    move_result = {
        "gold_collected": False,
        "new_pos": old_pos,
        "success": False,
        "failure_reason": None,
    }

    reason = blocked_reason(world_map, new_pos)
    if reason is not None:
        move_result["failure_reason"] = reason
        return move_result

    entity = world_map[old_pos[0], old_pos[1]]
    target = world_map[new_pos[0], new_pos[1]]
    world_map[old_pos[0], old_pos[1]] = VOID
    world_map[new_pos[0], new_pos[1]] = entity

    move_result["new_pos"] = new_pos
    move_result["success"] = True

    if target == GOLD:
        move_result["gold_collected"] = True

    return move_result

# Charge algorithmique avancée

- `valid_directions` *(bonus)* : filtre **en amont** les directions invalides (bord de carte, ennemi, obstacle) avant l'appel au LLM — activable via `prefilter_directions=True` dans la game loop.
- `direction_analysis` *(amélioration)* : lookahead à 1 coup — pour chaque direction praticable, distances résultantes vers l'or et l'ennemi les plus proches. L'algorithme **informe**, le LLM **décide** (cf. justification de traction dans les *Analyses* et le `README.md`).

In [18]:
def valid_directions(world_map: np.ndarray, player_position):
    """Bonus : filtre en amont les directions invalides (bord, ennemi, obstacle)
    avant de solliciter le LLM. Activable via prefilter_directions=True."""
    r, c = player_position
    return [
        name for name, (d_row, d_col) in MOVES.items()
        if allowed_move(world_map, (r + d_row, c + d_col))
    ]

In [19]:
def direction_analysis(world_map: np.ndarray, player_position):
    """Amélioration : lookahead à 1 coup.

    Pour chaque direction praticable, distances résultantes vers l'or et
    l'ennemi les plus proches. L'algorithme informe, le LLM décide."""
    gold_positions = localize(world_map, GOLD)
    enemy_positions = localize(world_map, ENEMY)
    r, c = player_position

    analysis = {}
    for name, (d_row, d_col) in MOVES.items():
        new_pos = (r + d_row, c + d_col)
        if not allowed_move(world_map, new_pos):
            continue
        gold_dist = compute_distance(gold_positions, np.array(new_pos))
        enemy_dist = compute_distance(enemy_positions, np.array(new_pos))
        analysis[name] = {
            "closest_gold_distance": float(np.min(gold_dist)) if len(gold_dist) > 0 else 999,
            "closest_enemy_distance": float(np.min(enemy_dist)) if len(enemy_dist) > 0 else 999,
        }
    return analysis

# Moteur de décision

In [20]:
def decide(player_perception: dict) -> PlayerDecision | None:
    prompt = f"""
    # Context
    - I am a player on a grid map and I want to collect gold.

    # Objective
    - Give me the single move that follows the shortest path toward the closest gold.

    # Rules
    - You must avoid enemies: never choose a direction whose adjacent cell contains ENEMY.
    - If "allowed_directions" is provided, you must pick a direction from this list only.
    - If "last_move_failed" is not null, the previous move failed for the given reason: do not repeat it.
    - Use "move_history" to detect and break oscillations: do not endlessly go back and forth.
    - If "direction_analysis" is provided, it gives for each walkable direction the resulting distances after that move: prefer moves that reduce the gold distance while keeping the enemy at a safe distance.

    # Player perception
    {player_perception} """

    # print(prompt)
    print(player_perception)

    response = client.beta.chat.completions.parse(
            model    = MODEL,
            messages = [{"role": "user", "content": prompt}],
            temperature=0,
            response_format=PlayerDecision
        )

    return response.choices[0].message.parsed or None

# Game loop (simulation)

In [21]:
initial_map = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [0, 1, 0, 0, 2, 0, 3],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 3],
    [0, 0, 0, 0, 0, 0, 0], 
])

In [ ]:
HISTORY_SIZE = 10 

def game_loop(world_map: np.ndarray, max_turns = 10,
              use_history = True, use_lookahead = True, prefilter_directions = False):
    world_map = world_map.copy()

    move_history = []          # historique des déplacements (fourni au LLM)
    last_move_failure = None   # signalement de l'échec du dernier déplacement

    for turn in range(max_turns):
        print(f"\n =================== [Turn {turn + 1}] ===================")

        show_map(world_map)

        player_pos = localize(world_map, PLAYER)[0]

        p = perception(world_map)
        p["last_move_failed"] = last_move_failure
        if use_history:
            p["move_history"] = move_history[-HISTORY_SIZE:]
        if use_lookahead:
            p["direction_analysis"] = direction_analysis(world_map, player_pos)
        if prefilter_directions:
            p["allowed_directions"] = valid_directions(world_map, player_pos)

        decision: PlayerDecision | None = decide(p)

        if decision is None:
            print("\t → No decision returned by the LLM")
            continue

        direction = decision.direction.value
        print(f"\t → LLM decision: {direction}")

        d_row, d_col = MOVES[direction]
        new_pos = (player_pos[0] + d_row, player_pos[1] + d_col)
        move_result = move(world_map, player_pos, new_pos)

        move_history.append({
            "turn": turn + 1,
            "direction": direction,
            "success": move_result["success"],
        })

        if move_result["success"]:
            last_move_failure = None
        else:
            last_move_failure = {
                "direction": direction,
                "reason": move_result["failure_reason"],
            }
            print(f"\t → Move failed: {move_result['failure_reason']}")

        if move_result["gold_collected"]:
            print("\n >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> FOUND GOLD ! <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< \n")
            break

In [23]:
# Run principal : anglais + historique + signalement des échecs + lookahead
game_loop(world_map=initial_map, max_turns=10)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [3.0], 'enemy_count': 1, 'closest_enemy_distance': 3.0, 'gold_distances': [5.0, 6.4], 'closest_gold_distance': 5.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None, 'move_history': [], 'direction_analysis': {'UP': {'closest_gold_distance': 5.1, 'closest_enemy_distance': 3.16}, 'DOWN': {'closest_gold_distance': 5.1, 'closest_enemy_distance': 3.16}, 'LEFT': {'closest_gold_distance': 6.0, 'closest_enemy_distance': 4.0}, 'RIGHT': {'closest_gold_distance': 4.0, 'closest_enemy_distance': 2.0}}}
	 → LLM decision: RIGHT

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'e

## Expérience A/B : impact de l'historique des déplacements

Deux runs comparatifs (sans lookahead ni filtrage pour isoler la variable) : d'abord **sans** historique, puis **avec** (cf. *Analyses*).

In [24]:
# Sans historique (comportement de référence, sujet aux oscillations)
game_loop(world_map=initial_map, max_turns=10, use_history=False, use_lookahead=False)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [3.0], 'enemy_count': 1, 'closest_enemy_distance': 3.0, 'gold_distances': [5.0, 6.4], 'closest_gold_distance': 5.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None}
	 → LLM decision: RIGHT

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [2.0], 'enemy_count': 1, 'closest_enemy_distance': 2.0, 'gold_distances': [4.0, 5.66], 'closest_gold_distance': 4.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None}
	 → LLM decision: RIGHT

 ===================

In [25]:
# Avec historique (le LLM peut détecter et casser les oscillations)
game_loop(world_map=initial_map, max_turns=10, use_history=True, use_lookahead=False)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [3.0], 'enemy_count': 1, 'closest_enemy_distance': 3.0, 'gold_distances': [5.0, 6.4], 'closest_gold_distance': 5.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None, 'move_history': []}
	 → LLM decision: RIGHT

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [2.0], 'enemy_count': 1, 'closest_enemy_distance': 2.0, 'gold_distances': [4.0, 5.66], 'closest_gold_distance': 4.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None, 'move_history': [{'turn':

## Bonus : filtrage actif des directions invalides en amont de l'appel LLM

In [26]:
# Bonus : le LLM ne choisit plus que parmi les directions valides
game_loop(world_map=initial_map, max_turns=10, prefilter_directions=True)


 =================== [Turn 1] ===================
·	·	·	·	·	·	·
·	👤	·	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	💰
·	·	·	·	·	·	·
{'enemies_within_radius_3': 1, 'enemy_distances': [3.0], 'enemy_count': 1, 'closest_enemy_distance': 3.0, 'gold_distances': [5.0, 6.4], 'closest_gold_distance': 5.0, 'closest_gold_direction': ['RIGHT'], 'adjacent_cells': {'UP': 'EMPTY', 'DOWN': 'EMPTY', 'LEFT': 'EMPTY', 'RIGHT': 'EMPTY'}, 'last_move_failed': None, 'move_history': [], 'direction_analysis': {'UP': {'closest_gold_distance': 5.1, 'closest_enemy_distance': 3.16}, 'DOWN': {'closest_gold_distance': 5.1, 'closest_enemy_distance': 3.16}, 'LEFT': {'closest_gold_distance': 6.0, 'closest_enemy_distance': 4.0}, 'RIGHT': {'closest_gold_distance': 4.0, 'closest_enemy_distance': 2.0}}, 'allowed_directions': ['UP', 'DOWN', 'LEFT', 'RIGHT']}
	 → LLM decision: RIGHT

 =================== [Turn 2] ===================
·	·	·	·	·	·	·
·	·	👤	·	👹	·	💰
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	·	·	·	·
·	·	·	

# Analyses (réponses au sujet) (Rédigé avec Claude mais ma validation)

## 1. Suppression des biais liés à la langue

Le contrat (`UP/DOWN/LEFT/RIGHT`), les clés de perception (`closest_gold_distance`, `adjacent_cells`…) et le prompt sont passés intégralement en anglais. Les cases adjacentes sont désormais décrites par des labels sémantiques (`EMPTY`, `ENEMY`, `WALL`…) plutôt que par des emojis.

**Est-ce que cela a un impact (à modèle équivalent) ?** Oui, modeste mais réel. Les tokens `UP/DOWN/LEFT/RIGHT` sont beaucoup plus représentés dans les corpus d'entraînement que `HAUT/BAS/GAUCHE/DROITE` : l'association direction ↔ sémantique spatiale est plus fiable. Surtout, la version d'origine mélangeait les langues (prompt français, clés de perception en anglais approximatif, valeurs mi-françaises mi-emojis), ce qui ajoutait une friction d'interprétation à chaque tour. Après bascule, les décisions sont plus cohérentes avec `closest_gold_direction` ; sur un scénario aussi court, la trajectoire reste globalement similaire — le gain est une réduction du bruit décisionnel, pas un changement de stratégie.

## 2. Historique des déplacements

`move_history` (tour, direction, succès — `HISTORY_SIZE` dernières entrées) est injecté dans la perception, accompagné d'une règle anti-oscillation dans le prompt.

**Est-ce que cela influence positivement le résultat ?** Oui. Le mode d'échec principal du run de base était la boucle `HAUT`/`BAS` devant l'ennemi (tours 3 à 10 : le joueur n'atteint jamais l'or). Sans mémoire, chaque tour est décidé isolément et le LLM reproduit indéfiniment le même arbitrage. Avec l'historique, le cycle devient visible dans la perception et le LLM peut en sortir (contournement par `DOWN` puis `RIGHT`). Limite : un historique trop long dilue le signal dans le prompt, d'où le plafond `HISTORY_SIZE = 10`.

## 3. Perception des échecs de déplacement

`move()` retourne désormais `success` et `failure_reason` (`MAP_EDGE`, `ENEMY`, `OBSTACLE`). La game loop mémorise `last_move_failed = {direction, reason}` et l'injecte dans la perception du tour suivant ; l'information est remise à `None` dès qu'un déplacement réussit.

**Conformité à la définition de la charge algorithmique :** l'algorithme se contente de *constater et d'informer* — il est le seul à savoir de façon fiable pourquoi un déplacement a échoué. Le LLM garde l'entière responsabilité de la décision : la règle du prompt (« do not repeat it ») lui permet d'exploiter le signal, par exemple ne pas re-proposer `RIGHT` juste après un échec `ENEMY` sur `RIGHT`.

## 4. Amélioration : lookahead à 1 coup (`direction_analysis`)

Pour chaque direction praticable, l'algorithme fournit les distances résultantes vers l'or et l'ennemi les plus proches *après* ce déplacement.

**Justification de traction (3 lignes) :** cette amélioration bascule vers la charge algorithmique le calcul géométrique post-déplacement, tâche où le LLM est faible et l'algorithme infaillible. Le LLM conserve l'intégralité de l'arbitrage stratégique — trancher entre « se rapprocher de l'or » et « rester loin de l'ennemi ». La traction reste donc équilibrée : l'algorithme informe plus finement, mais ne choisit toujours rien.

## Bonus : filtrage actif des directions invalides (`prefilter_directions=True`)

`valid_directions` retire bords/ennemis/obstacles **avant** l'appel, et le prompt impose de choisir dans `allowed_directions`.

**Impact sur la traction algo/LLM :** par rapport à la version de base (signalement *a posteriori*), l'algorithme passe d'un rôle **informatif** à un rôle **normatif** : il contraint l'espace d'action *a priori*. Une partie du périmètre décisionnel bascule côté algorithmique — le LLM ne peut plus produire de coup invalide, sa robustesse devient donc moins critique et sa valeur se concentre sur le seul choix stratégique parmi des options garanties sûres. On gagne des tours (aucun coup perdu contre un mur ou un ennemi) au prix d'un LLM plus « anecdotique » sur la sécurité des déplacements : la traction se déplace nettement vers l'algorithme.